# KaroSpace Pseudobulk DE Audit

This notebook mirrors the current KaroSpace pseudobulk DE workflow for one selected contrast.

Current workflow:
- If `GROUPBY` is a section metadata column and its value is fixed within every replicate, KaroSpace uses the sample-metadata model: one pseudobulk sample per replicate and design `~ _pb_group`.
- Otherwise KaroSpace uses the cell/category model: one pseudobulk sample per replicate x annotation category and design `~ _pb_replicate + _pb_group`.
- For both paths, genes with low total pseudobulk counts are removed before fitting; the selected contrast is then run on the full fitted DESeq2 object with `independent_filter=False` when supported.
- `MIN_PCT_EXPRESSED` is applied to result rows after `DeseqStats`, matching the production helper that filters the returned DESeq2 table.
- `%A` and `%B` are not DESeq2 outputs. They are fractions of original cells in the source and reference masks with count > 0 (`pct_source` and `pct_reference`) and are compacted to the same precision used in KaroSpace JSON.

Use it to inspect one comparison before it is formatted for the UI/export payload.


In [ ]:
from pathlib import Path

# Inputs
H5AD_PATH = "./data.h5ad"
GROUPBY = "Anno_L1_curated"  # KaroSpace annotation_key / main cell annotation
REPLICATE = "meta_sample_id"  # KaroSpace section_key / replicate column
SOURCE = "Astrocyte"  # group A
REFERENCE = "B cell"  # group B, or None to compare SOURCE vs balanced rest

# Columns passed to KaroSpace as section metadata. If GROUPBY is here and fixed within each
# replicate, the production workflow uses the sample-metadata pseudobulk model.
SECTION_METADATA = []
SECTION_METADATA_EXTRA = []
FORCE_SAMPLE_METADATA_MODEL = (
    None  # None = auto like KaroSpace, True/False = override for audit
)

# Pseudobulk parameters
COUNTS_LAYER = "counts"  # None uses adata.X
MIN_CELL_COUNTS = (
    0  # API/CLI: pseudobulk_min_cell_counts / --pseudobulk-min-cell-counts
)
MIN_GENE_COUNTS = (
    0  # API/CLI: pseudobulk_min_gene_counts / --pseudobulk-min-gene-counts
)
MIN_CELLS = 20  # API/CLI: pseudobulk_min_cells_per_pseudobulk / --pseudobulk-min-cells-per-pseudobulk
MIN_REPLICATES = 2
MIN_PCT_EXPRESSED = 0  # 0.10 or 10 both mean 10 percent
P_ADJUST_METHOD = "fdr_bh"
FIT_TYPE = "parametric"  # KaroSpace default; use "mean" if that is what your run used
N_CPUS = 4

# Display thresholds, not part of DESeq2 fitting
PADJ_CUTOFF = 0.05
LOG2FC_CUTOFF = 1

# Optional selected-gene audit. Leave empty to use the contrast-specific MIN_PCT_EXPRESSED filter.
SELECTED_GENES = []

In [ ]:
import inspect
import json
import math
import warnings

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import rankdata
from statsmodels.stats.multitest import multipletests

try:
    from pydeseq2.dds import DeseqDataSet
    from pydeseq2.ds import DeseqStats
except ImportError as exc:
    raise ImportError(
        "Install pydeseq2 in this environment before running the audit notebook"
    ) from exc


def normalize_pct_threshold(value):
    value = float(value)
    if value < 0:
        raise ValueError("MIN_PCT_EXPRESSED must be >= 0")
    return value / 100.0 if value > 1 else value


def adjust_pvalues(pvalues, method="fdr_bh"):
    method = str(method or "fdr_bh").strip().lower().replace("-", "_")
    pvals = np.asarray(pvalues, dtype=float)
    out = np.full(pvals.shape, np.nan, dtype=float)
    mask = np.isfinite(pvals)
    if not mask.any():
        return out
    vals = np.clip(pvals[mask], 0.0, 1.0)
    if method in {"none", "raw", "pvalue", "pvalues"}:
        adjusted = vals
    elif method in {"bonferroni", "bonf"}:
        adjusted = np.minimum(vals * vals.size, 1.0)
    elif method in {"holm", "holm_bonferroni"}:
        adjusted = multipletests(vals, method="holm")[1]
    else:
        adjusted = multipletests(vals, method="fdr_bh")[1]
    adjusted[(adjusted == 0) & np.isfinite(adjusted)] = np.nextafter(0.0, 1.0)
    out[mask] = adjusted
    return out


def compact_json_float(value, significant_digits=6):
    if value is None:
        return None
    try:
        numeric = float(value)
    except (TypeError, ValueError):
        return None
    if not math.isfinite(numeric):
        return None
    text = format(numeric, f".{significant_digits}g")
    return float(text)


def to_dense_counts(matrix):
    dense = matrix.toarray() if sparse.issparse(matrix) else np.asarray(matrix)
    dense = np.asarray(dense, dtype=np.float64)
    dense[~np.isfinite(dense)] = 0
    dense[dense < 0] = 0
    return np.rint(dense).astype(np.int64, copy=False)


def cell_count_mask(matrix, min_cell_counts):
    threshold = max(0, int(min_cell_counts))
    if threshold == 0:
        return np.ones(int(matrix.shape[0]), dtype=bool)
    totals = (
        np.asarray(matrix.sum(axis=1)).ravel()
        if sparse.issparse(matrix)
        else np.asarray(matrix).sum(axis=1)
    )
    totals = np.asarray(totals, dtype=float)
    return np.isfinite(totals) & (totals >= threshold)


def filter_pseudobulk_genes(counts, gene_names, min_gene_counts):
    threshold = max(0, int(min_gene_counts))
    gene_names = [str(g) for g in gene_names]
    if threshold == 0 or counts.size == 0:
        return counts, gene_names
    totals = np.asarray(counts, dtype=float).sum(axis=0)
    keep = np.isfinite(totals) & (totals >= threshold)
    return np.asarray(counts)[:, keep], [
        gene for gene, ok in zip(gene_names, keep) if ok
    ]


def shared_category_design_rank(metadata):
    # Match karospace.pseudobulk._shared_category_design_rank: intercept plus
    # treatment-coded replicate and group factors. Keeping every group dummy with
    # an intercept adds one redundant column and falsely reports rank deficiency.
    reps = pd.get_dummies(
        metadata["_pb_replicate"].astype(str), drop_first=True, dtype=float
    )
    groups = pd.get_dummies(
        metadata["_pb_group"].astype(str), drop_first=True, dtype=float
    )
    design = pd.concat(
        [pd.Series(1.0, index=metadata.index, name="Intercept"), reps, groups], axis=1
    )
    matrix = design.to_numpy(dtype=float)
    rank = np.linalg.matrix_rank(matrix)
    return rank, matrix.shape[1], design.columns.tolist()


def sample_metadata_design_rank(metadata):
    # Match karospace.pseudobulk._sample_metadata_design_rank.
    groups = pd.get_dummies(
        metadata["_pb_group"].astype(str), drop_first=True, dtype=float
    )
    design = pd.concat(
        [pd.Series(1.0, index=metadata.index, name="Intercept"), groups], axis=1
    )
    matrix = design.to_numpy(dtype=float)
    rank = np.linalg.matrix_rank(matrix)
    return rank, matrix.shape[1], design.columns.tolist()


def positive_fraction(matrix):
    if sparse.issparse(matrix):
        n = matrix.shape[0]
        if n == 0:
            return np.zeros(matrix.shape[1], dtype=float)
        return np.asarray(matrix.getnnz(axis=0)).ravel() / float(n)
    arr = np.asarray(matrix)
    if arr.shape[0] == 0:
        return np.zeros(arr.shape[1], dtype=float)
    return (arr > 0).mean(axis=0)


def expression_prefilter_gene_names(
    count_matrix,
    source_mask,
    reference_mask,
    all_gene_names,
    fitted_gene_names,
    min_pct,
):
    threshold = normalize_pct_threshold(min_pct)
    fitted_gene_names = [str(g) for g in fitted_gene_names]
    if threshold <= 0:
        return fitted_gene_names
    source_frac = positive_fraction(count_matrix[source_mask])
    reference_frac = positive_fraction(count_matrix[reference_mask])
    all_gene_names = np.asarray([str(g) for g in all_gene_names])
    expressed = set(
        all_gene_names[
            (source_frac >= threshold) | (reference_frac >= threshold)
        ].tolist()
    )
    return [gene for gene in fitted_gene_names if gene in expressed]


def filter_deseq2_result_genes(results_df, gene_names=None):
    if gene_names is None:
        return results_df.copy()
    ordered = [str(g) for g in gene_names]
    if not ordered:
        return results_df.iloc[0:0].copy()
    available = results_df.index.astype(str)
    selected = [gene for gene in ordered if gene in set(available)]
    if not selected:
        return results_df.iloc[0:0].copy()
    return results_df.reindex(selected).dropna(how="all")


def run_deseq2_contrast(dds, contrast_vector, gene_names=None, n_cpus=1):
    kwargs = {
        "dds": dds,
        "contrast": contrast_vector,
        "quiet": True,
        "n_cpus": max(1, int(n_cpus)),
    }
    if "independent_filter" in inspect.signature(DeseqStats.__init__).parameters:
        kwargs["independent_filter"] = False
    stat_res = DeseqStats(**kwargs)
    if hasattr(stat_res, "summary"):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            stat_res.summary()
    results = getattr(stat_res, "results_df", None)
    if results is None:
        return pd.DataFrame()
    return filter_deseq2_result_genes(results.copy(), gene_names)


def format_p(value):
    if value is None or not np.isfinite(value):
        return "NA"
    if value == 0:
        return "0"
    if value < 1e-3:
        return f"{value:.2e}"
    return f"{value:.4f}"

## Load, Choose Workflow, And Aggregate

The next cell loads the AnnData object, selects the same pseudobulk model KaroSpace would use for `GROUPBY`, and builds the pseudobulk count matrix used by DESeq2.


In [15]:
adata = ad.read_h5ad(H5AD_PATH)
print(adata)

if GROUPBY not in adata.obs:
    raise KeyError(f"GROUPBY column {GROUPBY!r} is not present in adata.obs")
if REPLICATE not in adata.obs:
    raise KeyError(f"REPLICATE column {REPLICATE!r} is not present in adata.obs")

if COUNTS_LAYER:
    if COUNTS_LAYER in adata.layers:
        count_matrix = adata.layers[COUNTS_LAYER]
        counts_layer_used = str(COUNTS_LAYER)
        count_warning = None
    else:
        count_matrix = adata.X
        counts_layer_used = "X"
        count_warning = f"counts layer {COUNTS_LAYER!r} not found; using adata.X"
        warnings.warn(count_warning)
else:
    count_matrix = adata.X
    counts_layer_used = "X"
    count_warning = None
if count_matrix.shape != (adata.n_obs, adata.n_vars):
    raise ValueError("Selected count matrix shape does not match adata")

obs = adata.obs
rep_values = obs[REPLICATE].astype(str)
col = obs[GROUPBY]
if pd.api.types.is_numeric_dtype(col):
    raise ValueError(
        f"GROUPBY column {GROUPBY!r} is numeric; pseudobulk DE expects categorical groups"
    )
if not isinstance(col.dtype, pd.CategoricalDtype):
    col = col.astype("category")
categories = [str(category) for category in col.cat.categories]
group_values = col.astype(str)
valid = rep_values.notna().to_numpy() & group_values.notna().to_numpy()
valid &= np.asarray(col.cat.codes.to_numpy() >= 0, dtype=bool)
valid &= cell_count_mask(count_matrix, MIN_CELL_COUNTS)
print(f"counts source: {counts_layer_used}")
print(
    f"valid cells after cell-level count and metadata filters: {int(valid.sum()):,} / {adata.n_obs:,}"
)

section_metadata_columns = {str(c) for c in (SECTION_METADATA or [])} | {
    str(c) for c in (SECTION_METADATA_EXTRA or [])
}


def should_use_sample_metadata_model():
    if FORCE_SAMPLE_METADATA_MODEL is not None:
        return bool(FORCE_SAMPLE_METADATA_MODEL)
    if GROUPBY not in section_metadata_columns:
        return False
    valid_indices = np.flatnonzero(valid)
    if valid_indices.size == 0:
        return False
    rep_valid = rep_values.iloc[valid_indices].to_numpy(dtype=str)
    group_valid_values = group_values.iloc[valid_indices].to_numpy(dtype=str)
    per_replicate_groups = {}
    mixed_replicates = set()
    for rep_value, group_value in zip(rep_valid, group_valid_values):
        previous = per_replicate_groups.get(rep_value)
        if previous is None:
            per_replicate_groups[rep_value] = group_value
        elif previous != group_value:
            mixed_replicates.add(rep_value)
    if mixed_replicates:
        preview = sorted(mixed_replicates)[:20]
        print(
            "GROUPBY is section metadata but is not fixed within every replicate; "
            "falling back to the replicate-by-category model."
        )
        display(pd.DataFrame({"mixed_replicate": preview}))
        return False
    return True


sample_metadata_model = should_use_sample_metadata_model()
workflow_name = "sample_metadata" if sample_metadata_model else "shared_all_category"
print(f"pseudobulk workflow: {workflow_name}")

valid_indices = np.flatnonzero(valid)
if valid_indices.size == 0:
    raise ValueError("No cells passed the cell-level filters")
rep_valid = rep_values.iloc[valid_indices].to_numpy(dtype=str)
group_valid_values = group_values.iloc[valid_indices].to_numpy(dtype=str)

if sample_metadata_model:
    per_replicate_groups = {}
    sample_keys = []
    sample_index = {}
    row_ids = np.empty(valid_indices.size, dtype=np.int64)
    for i, (rep_value, group_value) in enumerate(zip(rep_valid, group_valid_values)):
        if rep_value not in sample_index:
            sample_index[rep_value] = len(sample_keys)
            sample_keys.append(rep_value)
            per_replicate_groups[rep_value] = group_value
        row_ids[i] = sample_index[rep_value]
    rows = []
    meta_rows = []
    for replicate in sample_keys:
        mask = valid & (rep_values.to_numpy() == replicate)
        n_cells = int(mask.sum())
        rows.append(np.asarray(count_matrix[mask].sum(axis=0)).ravel())
        meta_rows.append(
            {
                "_pb_replicate": replicate,
                "_pb_group": per_replicate_groups[replicate],
                "n_cells": n_cells,
            }
        )
else:
    sample_keys = []
    sample_index = {}
    row_ids = np.empty(valid_indices.size, dtype=np.int64)
    for i, (rep_value, group_value) in enumerate(zip(rep_valid, group_valid_values)):
        key = (str(rep_value), str(group_value))
        if key not in sample_index:
            sample_index[key] = len(sample_keys)
            sample_keys.append(key)
        row_ids[i] = sample_index[key]
    rows = []
    meta_rows = []
    for replicate, group in sample_keys:
        mask = (
            valid
            & (rep_values.to_numpy() == replicate)
            & (group_values.to_numpy() == group)
        )
        n_cells = int(mask.sum())
        rows.append(np.asarray(count_matrix[mask].sum(axis=0)).ravel())
        meta_rows.append(
            {"_pb_replicate": replicate, "_pb_group": group, "n_cells": n_cells}
        )

if not rows:
    raise ValueError("No pseudobulk samples could be aggregated")

aggregate = to_dense_counts(np.vstack(rows))
pb_meta = pd.DataFrame(meta_rows)
pb_meta["_pb_group"] = pb_meta["_pb_group"].astype(str)
pb_meta["_pb_replicate"] = pb_meta["_pb_replicate"].astype(str)
pb_meta.index = [f"pb_{i}" for i in range(len(pb_meta))]
pb_meta.attrs["gene_names"] = [str(gene) for gene in adata.var_names]
print(
    f"pseudobulk samples before sample-cell filter: {aggregate.shape[0]:,}; genes before gene-count filter: {aggregate.shape[1]:,}"
)
display(
    pb_meta.groupby("_pb_group")
    .agg(
        n_pseudobulk=("_pb_group", "size"),
        n_replicates=("_pb_replicate", "nunique"),
        cells=("n_cells", "sum"),
    )
    .reindex(categories)
    .dropna(how="all")
)

AnnData object with n_obs × n_vars = 1384881 × 5101
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'run_id', 'sample_id', 'sample_label', 'xenium_output', 'instrument_or_flowcell', 'cassette_or_slide_id', 'run_date', 'cell_id', 'kmeans_split_id', 'n_counts', 'n_genes', 'leiden_1.5', 'leiden_2', 'leiden_2.5', 'leiden_3', 'leiden_3.5', 'leiden_4.0', 'CellCharter_5', 'CellCharter_10', 'CellCharter_15', 'CellCharter_20', 'CellCharter_25', 'CellCharter_30', 'CellCharter_35', 'CellCharter_40', 'CellCharter_45', 'CellCharter_50', 'meta_sample_id', 'sample_name', 'stage', 'condition', 'day_of_sacrifice', 'score_sacrifice', 'region', 'sex', 'model', 'polygon_label', 'polygon_id', 'polygon_remove', 'polygon_remove_0', 'polygon_remove_1', 'Anno_L1', 'CellCharter_60', 

,n_pseudobulk,n_replicates,cells
_pb_group,,,
Astrocyte,158,158,141599
B cell,146,146,13245
DC,158,158,60927
Doublet,157,157,3078
Endothelial,158,158,112244
Ependymal cell,158,158,9158
Epithelial,70,70,124
Fibroblast,158,158,143411
Muscle cell,13,13,180


In [ ]:
required_min_replicates = max(2, int(MIN_REPLICATES))
source = str(SOURCE)
reference = None if REFERENCE is None else str(REFERENCE)
rest_reference = "__rest__"
reference_key = rest_reference if reference is None else reference
reference_label = "balanced rest" if reference is None else reference

if source not in categories:
    raise ValueError(f"SOURCE {source!r} is not a category of {GROUPBY!r}")
if reference is not None and reference not in categories:
    raise ValueError(f"REFERENCE {reference!r} is not a category of {GROUPBY!r}")

sufficient_pb = pb_meta[pb_meta["n_cells"] >= int(MIN_CELLS)]
retained_categories = [
    category
    for category in categories
    if sufficient_pb.loc[
        sufficient_pb["_pb_group"] == category, "_pb_replicate"
    ].nunique()
    >= required_min_replicates
]
omitted_categories = [
    category for category in categories if category not in retained_categories
]
if len(retained_categories) < 2:
    raise ValueError(
        f"Fewer than two categories have >= {required_min_replicates} replicate pseudobulks "
        f"with >= {int(MIN_CELLS)} cells"
    )
if source not in retained_categories:
    raise ValueError(
        f"SOURCE {source!r} has fewer than {required_min_replicates} qualifying replicate pseudobulks"
    )
if reference is not None and reference not in retained_categories:
    raise ValueError(
        f"REFERENCE {reference!r} has fewer than {required_min_replicates} qualifying replicate pseudobulks"
    )

model_mask = pb_meta["_pb_group"].isin(retained_categories) & (
    pb_meta["n_cells"] >= int(MIN_CELLS)
)
model_positions = np.flatnonzero(model_mask.to_numpy())
model_counts = aggregate[model_positions]
model_meta = pb_meta.iloc[model_positions].copy()
model_meta.attrs["gene_names"] = [str(gene) for gene in adata.var_names]

if sample_metadata_model:
    rank, n_cols, design_columns = sample_metadata_design_rank(model_meta)
    design_label = f"~ {GROUPBY}"
else:
    rank, n_cols, design_columns = shared_category_design_rank(model_meta)
    design_label = f"~ {REPLICATE} + {GROUPBY}"
residual_df = int(len(model_meta) - rank)
if rank < n_cols or residual_df <= 0:
    raise ValueError(
        f"Design {design_label} is rank deficient or has no residual df: rank {rank}/{n_cols}; residual df {residual_df}"
    )

pair_counts, pair_gene_names = filter_pseudobulk_genes(
    model_counts, adata.var_names, MIN_GENE_COUNTS
)
pair_meta = model_meta.copy()
pair_meta.attrs["gene_names"] = pair_gene_names
if not pair_gene_names:
    raise ValueError(f"No genes meet MIN_GENE_COUNTS={max(0, int(MIN_GENE_COUNTS))}")

model_keys = set(
    zip(pair_meta["_pb_replicate"].astype(str), pair_meta["_pb_group"].astype(str))
)
if sample_metadata_model:
    model_replicates = set(pair_meta["_pb_replicate"].astype(str))
    model_cell_mask = valid & rep_values.astype(str).isin(model_replicates).to_numpy()
else:
    model_cell_mask = valid & np.fromiter(
        (
            (str(rep_value), str(group_value)) in model_keys
            for rep_value, group_value in zip(
                rep_values.to_numpy(), group_values.to_numpy()
            )
        ),
        dtype=bool,
        count=adata.n_obs,
    )

reference_groups = (
    [category for category in retained_categories if category != source]
    if reference is None
    else [reference]
)
source_cell_mask = model_cell_mask & (group_values.to_numpy() == source)
reference_cell_mask = (
    model_cell_mask & (group_values.to_numpy() != source)
    if reference is None
    else model_cell_mask & (group_values.to_numpy() == reference)
)

if sample_metadata_model:
    replicate_counts = [
        int(
            pair_meta.loc[pair_meta["_pb_group"] == category, "_pb_replicate"].nunique()
        )
        for category in [source, *reference_groups]
    ]
    contrast_replicates = min(replicate_counts) if replicate_counts else 0
else:
    if reference is None:
        source_reps = set(
            pair_meta.loc[pair_meta["_pb_group"] == source, "_pb_replicate"].astype(str)
        )
        paired_reps = sorted(
            rep_value
            for rep_value in source_reps
            if any(
                rep_value
                in set(
                    pair_meta.loc[
                        pair_meta["_pb_group"] == ref, "_pb_replicate"
                    ].astype(str)
                )
                for ref in reference_groups
            )
        )
    else:
        paired_reps = sorted(
            set(
                pair_meta.loc[pair_meta["_pb_group"] == source, "_pb_replicate"].astype(
                    str
                )
            )
            & set(
                pair_meta.loc[
                    pair_meta["_pb_group"] == reference, "_pb_replicate"
                ].astype(str)
            )
        )
    contrast_replicates = len(paired_reps)

if contrast_replicates < required_min_replicates:
    raise ValueError(
        f"Contrast has {contrast_replicates} replicate(s); need >= {required_min_replicates}"
    )

print(f"retained categories for shared model: {retained_categories}")
if omitted_categories:
    print(f"omitted categories: {omitted_categories}")
print(
    f"contrast: {source} vs {reference_label}; production key: {reference_key}; contrast replicate count: {contrast_replicates}"
)
print(
    f"model pseudobulk samples: {pair_counts.shape[0]:,}; genes after gene-count filter: {pair_counts.shape[1]:,}"
)
print(
    f"design {design_label}: rank {rank}/{n_cols}; residual df {residual_df}; columns: {design_columns}"
)
print(
    f"source cells: {int(source_cell_mask.sum()):,}; reference cells: {int(reference_cell_mask.sum()):,}"
)
display(pair_meta.sort_values(["_pb_replicate", "_pb_group"]))

In [ ]:
# Fit DESeq2 on the retained production model, then run the selected contrast.
counts_df = pd.DataFrame(pair_counts, index=pair_meta.index, columns=pair_gene_names)
meta_df = pair_meta.copy()
meta_df["_pb_group"] = pd.Categorical(
    meta_df["_pb_group"].astype(str), categories=retained_categories
)

if sample_metadata_model:
    design_factors = ["_pb_group"]
    design = "~ _pb_group"
else:
    meta_df["_pb_replicate"] = pd.Categorical(meta_df["_pb_replicate"].astype(str))
    design_factors = ["_pb_replicate", "_pb_group"]
    design = "~ _pb_replicate + _pb_group"

try:
    dds = DeseqDataSet(
        counts=counts_df,
        metadata=meta_df,
        design=design,
        fit_type=FIT_TYPE,
        quiet=True,
        n_cpus=max(1, int(N_CPUS)),
    )
except TypeError:
    dds = DeseqDataSet(
        counts=counts_df,
        clinical=meta_df,
        design_factors=design_factors,
        fit_type=FIT_TYPE,
        refit_cooks=True,
        n_cpus=max(1, int(N_CPUS)),
    )

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=RuntimeWarning)
    dds.deseq2()

if reference is None:
    if not hasattr(dds, "contrast"):
        raise RuntimeError(
            "This pydeseq2 version does not expose DeseqDataSet.contrast for balanced-rest contrasts"
        )
    contrast_vector = np.mean(
        [dds.contrast("_pb_group", source, ref) for ref in reference_groups], axis=0
    )
else:
    if hasattr(dds, "contrast"):
        contrast_vector = dds.contrast("_pb_group", source, reference)
    else:
        contrast_vector = ["_pb_group", source, reference]

selected_genes = [str(g) for g in SELECTED_GENES if str(g) in pair_gene_names]
if selected_genes:
    test_gene_names = selected_genes
else:
    test_gene_names = expression_prefilter_gene_names(
        count_matrix,
        source_cell_mask,
        reference_cell_mask,
        adata.var_names,
        pair_gene_names,
        MIN_PCT_EXPRESSED,
    )

raw_results = run_deseq2_contrast(
    dds, contrast_vector, gene_names=test_gene_names, n_cpus=1
)
contrast_terms = getattr(dds, "variables", None)
if contrast_terms is None:
    contrast_terms = getattr(dds, "factor_storage", None)
print(f"DESeq2 design: {design}; contrast terms: {contrast_terms}")
print(
    f"genes fitted before pct filter: {len(pair_gene_names):,}; genes reported after pct filter: {len(raw_results):,}"
)
raw_results.head()

In [ ]:
# Format the table using the same post-processing as KaroSpace.
def _base_result_payload():
    return {
        "method": (
            "pseudobulk-deseq2-sample-metadata"
            if sample_metadata_model
            else "pseudobulk-deseq2"
        ),
        "p_adjust_method": str(P_ADJUST_METHOD or "fdr_bh")
        .strip()
        .lower()
        .replace("-", "_"),
        "min_pct_expressed": normalize_pct_threshold(MIN_PCT_EXPRESSED),
        "padj_cutoff": float(PADJ_CUTOFF),
        "log2fc_cutoff": float(LOG2FC_CUTOFF),
        "table_top_n": 0,
        "n_source": int(source_cell_mask.sum()),
        "n_reference": int(reference_cell_mask.sum()),
        "n_replicates": int(contrast_replicates),
        "counts_layer": counts_layer_used,
        **({"warning": count_warning} if count_warning else {}),
        "model_formula": design_label,
        "model_categories": retained_categories,
        "contrast_reference": "balanced_rest",
        "contrast_type": (
            "balanced_rest" if reference is None else "category_vs_category"
        ),
        "min_pct_prefilter_gene_count": len(pair_gene_names),
        "min_pct_retained_gene_count": len(test_gene_names),
        "min_pct_removed_gene_count": int(
            max(0, len(pair_gene_names) - len(test_gene_names))
        ),
        "comparison": f"{source}_vs_{reference_key}",
        "source": source,
        "reference": reference_key,
        "reference_label": reference_label,
        "reference_groups": reference_groups,
    }


base_payload = _base_result_payload()
if raw_results.empty:
    karospace_result_payload = {
        "available": True,
        "genes": [],
        "log2foldchanges": [],
        "pvals": [],
        "pvals_adj": [],
        "scores": [],
        "pct_source": [],
        "pct_reference": [],
        "base_mean": [],
        **base_payload,
    }
else:
    work = raw_results.copy()
    work.index = work.index.astype(str)
    work["gene"] = work.index
    work["pvalue"] = pd.to_numeric(work.get("pvalue"), errors="coerce")
    work["padj"] = adjust_pvalues(
        work["pvalue"].to_numpy(dtype=float), method=P_ADJUST_METHOD
    )
    work["log2FoldChange"] = pd.to_numeric(work.get("log2FoldChange"), errors="coerce")
    work["baseMean"] = pd.to_numeric(work.get("baseMean"), errors="coerce")
    work["stat"] = pd.to_numeric(work.get("stat"), errors="coerce")

    gene_to_idx = {str(g): i for i, g in enumerate(adata.var_names.astype(str))}
    valid_gene_mask = work["gene"].isin(gene_to_idx)
    work = work.loc[valid_gene_mask].copy()
    gene_indices = [gene_to_idx[g] for g in work["gene"]]
    work["pct_source"] = positive_fraction(count_matrix[source_cell_mask])[gene_indices]
    work["pct_reference"] = positive_fraction(count_matrix[reference_cell_mask])[
        gene_indices
    ]

    finite_lfc = np.isfinite(work["log2FoldChange"].to_numpy())
    work = work.loc[finite_lfc].copy()
    work["_padj_sort"] = work["padj"].where(np.isfinite(work["padj"]), np.inf)
    work["_pvalue_sort"] = work["pvalue"].where(np.isfinite(work["pvalue"]), np.inf)
    work["_abs_lfc"] = work["log2FoldChange"].abs()
    work = work.sort_values(
        ["_padj_sort", "_pvalue_sort", "_abs_lfc", "gene"],
        ascending=[True, True, False, True],
    )

    karospace_result_payload = {
        "available": True,
        "genes": work["gene"].tolist(),
        "log2foldchanges": [compact_json_float(v, 6) for v in work["log2FoldChange"]],
        "pvals": [compact_json_float(v, 6) for v in work["pvalue"]],
        "pvals_adj": [compact_json_float(v, 6) for v in work["padj"]],
        "scores": [compact_json_float(v, 6) for v in work["stat"]],
        "pct_source": [compact_json_float(v, 5) for v in work["pct_source"]],
        "pct_reference": [compact_json_float(v, 5) for v in work["pct_reference"]],
        "base_mean": [compact_json_float(v, 6) for v in work["baseMean"]],
        **base_payload,
    }

de_table = pd.DataFrame(
    {
        "gene": karospace_result_payload["genes"],
        "baseMean": karospace_result_payload["base_mean"],
        "log2fc": karospace_result_payload["log2foldchanges"],
        "pvalue": karospace_result_payload["pvals"],
        "padj": karospace_result_payload["pvals_adj"],
        "score": karospace_result_payload["scores"],
        "pct_source": karospace_result_payload["pct_source"],
        "pct_reference": karospace_result_payload["pct_reference"],
    }
)

print(
    json.dumps(
        {
            k: v
            for k, v in karospace_result_payload.items()
            if k
            not in {
                "genes",
                "base_mean",
                "log2foldchanges",
                "pvals",
                "pvals_adj",
                "scores",
                "pct_source",
                "pct_reference",
            }
        },
        indent=2,
    )
)
de_table.head(20)

In [ ]:
# Threshold summaries matching the viewer logic.
padj_cutoff = float(PADJ_CUTOFF)
log2fc_cutoff = float(LOG2FC_CUTOFF)
ma_padj_cutoff = 0.1
stats_pass = (de_table["padj"] < padj_cutoff) & (
    de_table["log2fc"].abs() >= log2fc_cutoff
)
volcano_pass = stats_pass
ma_padj_pass = de_table["padj"] < ma_padj_cutoff
ma_pass = ma_padj_pass

print(
    f"MA red genes, adj. p < {ma_padj_cutoff}: {int(ma_pass.sum()):,} / {len(de_table):,}"
)
print(f"  grey, adj. p >= {ma_padj_cutoff}: {int((~ma_padj_pass).sum()):,}")
print(
    "Volcano/table passing genes, "
    f"padj < {padj_cutoff}, |log2FC| >= {log2fc_cutoff}: {int(volcano_pass.sum()):,} / {len(de_table):,}"
)
print(
    f"  positive/source-colored: {int((volcano_pass & (de_table['log2fc'] > 0)).sum()):,}"
)
print(
    f"  negative/reference-colored: {int((volcano_pass & (de_table['log2fc'] < 0)).sum()):,}"
)

display(de_table.loc[volcano_pass].head(50))

In [ ]:
# MA and volcano plots matching the viewer dot classes.
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

ax = axes[0]
ma_layers = [
    (~ma_padj_pass, "#b8b8b8", f"adj. p >= 0.1 ({int((~ma_padj_pass).sum()):,})", 0.55),
    (ma_pass, "#d94f4f", f"adj. p < 0.1 ({int(ma_pass.sum()):,})", 0.86),
]
for mask, color, label, alpha in ma_layers:
    if not mask.any():
        continue
    ax.scatter(
        de_table.loc[mask, "baseMean"],
        de_table.loc[mask, "log2fc"],
        c=color,
        s=18,
        alpha=alpha,
        linewidths=0,
        label=label,
    )
ax.set_xlabel("baseMean")
ax.set_ylabel("log2FC")
ax.set_title("MA plot")
ax.legend(frameon=False, fontsize=8, loc="best")
ax.grid(False)

ax = axes[1]
source_pass = volcano_pass & (de_table["log2fc"] > 0)
reference_pass = volcano_pass & (de_table["log2fc"] < 0)
volcano_layers = [
    (
        ~stats_pass,
        "#b8b8b8",
        f"fail adj. p/log2FC ({int((~stats_pass).sum()):,})",
        0.55,
    ),
    (
        reference_pass,
        "#4f82d9",
        f"{reference_label} ({int(reference_pass.sum()):,})",
        0.86,
    ),
    (source_pass, "#d94f4f", f"{source} ({int(source_pass.sum()):,})", 0.86),
]
safe_padj = de_table["padj"].fillna(1).clip(lower=np.nextafter(0.0, 1.0))
neg_log10_padj = -np.log10(safe_padj)
for mask, color, label, alpha in volcano_layers:
    if not mask.any():
        continue
    ax.scatter(
        de_table.loc[mask, "log2fc"],
        neg_log10_padj.loc[mask],
        c=color,
        s=18,
        alpha=alpha,
        linewidths=0,
        label=label,
    )
ax.set_xlabel("log2FC")
ax.set_ylabel("-log10 adjusted p")
ax.set_title("Volcano plot")
ax.legend(frameon=False, fontsize=8, loc="best")

plt.show()

In [ ]:
# Pairwise diagnostic: independent pairwise fit for SOURCE vs REFERENCE when applicable.
# This is not the exported KaroSpace result; it is a sanity check against the current fitted cohort.
if REFERENCE is None:
    print(
        "Pairwise diagnostic skipped because REFERENCE=None uses balanced-rest contrast."
    )
else:
    diag_mask = pair_meta["_pb_group"].isin([source, reference]).to_numpy()
    diag_counts = pair_counts[diag_mask]
    diag_meta = pair_meta.loc[diag_mask].copy()
    diag_meta["_pb_group"] = pd.Categorical(diag_meta["_pb_group"].astype(str))
    if sample_metadata_model:
        diag_design = "~ _pb_group"
        diag_design_factors = ["_pb_group"]
    else:
        diag_meta["_pb_replicate"] = pd.Categorical(
            diag_meta["_pb_replicate"].astype(str)
        )
        diag_design = "~ _pb_replicate + _pb_group"
        diag_design_factors = ["_pb_replicate", "_pb_group"]
    try:
        diag_dds = DeseqDataSet(
            counts=pd.DataFrame(
                diag_counts, index=diag_meta.index, columns=pair_gene_names
            ),
            metadata=diag_meta,
            design=diag_design,
            fit_type=FIT_TYPE,
            quiet=True,
            n_cpus=max(1, int(N_CPUS)),
        )
    except TypeError:
        diag_dds = DeseqDataSet(
            counts=pd.DataFrame(
                diag_counts, index=diag_meta.index, columns=pair_gene_names
            ),
            clinical=diag_meta,
            design_factors=diag_design_factors,
            fit_type=FIT_TYPE,
            quiet=True,
            n_cpus=max(1, int(N_CPUS)),
        )
    diag_dds.deseq2()
    if hasattr(diag_dds, "contrast"):
        diag_contrast = diag_dds.contrast("_pb_group", source, reference)
    else:
        diag_contrast = ["_pb_group", source, reference]
    diag_results = run_deseq2_contrast(
        diag_dds, diag_contrast, gene_names=test_gene_names, n_cpus=N_CPUS
    )
    diag = diag_results.rename(
        columns={
            "log2FoldChange": "pairwise_log2fc",
            "pvalue": "pairwise_pvalue",
            "padj": "pairwise_padj",
        }
    )
    joined = de_table.merge(
        diag[["pairwise_log2fc", "pairwise_pvalue", "pairwise_padj"]],
        left_on="gene",
        right_index=True,
        how="left",
    )
    joined["delta_log2fc"] = pd.to_numeric(
        joined["log2fc"], errors="coerce"
    ) - pd.to_numeric(joined["pairwise_log2fc"], errors="coerce")
    display(
        joined[
            [
                "gene",
                "log2fc",
                "pairwise_log2fc",
                "delta_log2fc",
                "padj",
                "pairwise_padj",
            ]
        ].head(30)
    )

In [ ]:
# Save the audited result table.
# out_csv = Path(H5AD_PATH).with_suffix(f".{GROUPBY}.{source}_vs_{reference_key}.pseudobulk_audit.csv")
# de_table.to_csv(out_csv, index=False)
# print(out_csv)

## Optional Pathway Enrichment Audit

Run the next cells after the DE table is available. The gene lists are built from the same formatted pseudobulk result used by KaroSpace, using `PADJ_CUTOFF`, `LOG2FC_CUTOFF`, `MIN_PCT_EXPRESSED`, and `PATHWAY_TOP_N`.


In [ ]:
# Pathway parameters matching exporter defaults
PATHWAY_ORGANISM = "Mouse"
PATHWAY_GMT = None  # None = Reactome via gseapy; or set to a GMT path/list of GMT paths
PATHWAY_TOP_N = 10
PATHWAY_MIN_OVERLAP = 3
PATHWAY_MAX_PATHWAY_SIZE = 500
PATHWAY_GSEA_PERMUTATIONS = 100
PATHWAY_SEED = 0

In [ ]:
from karospace.pathways import add_pathway_enrichment_to_pseudobulk_de

# Build the same nested pseudobulk_de shape used by the exporter and attach pathway_enrichment in place.
pathway_payload = json.loads(json.dumps(karospace_result_payload))
pseudobulk_de_for_pathway = {GROUPBY: {source: {reference_key: pathway_payload}}}
pathway_events = []

pathway_settings = add_pathway_enrichment_to_pseudobulk_de(
    pseudobulk_de_for_pathway,
    pathway_gmt=PATHWAY_GMT,
    top_n=int(PATHWAY_TOP_N),
    min_overlap=int(PATHWAY_MIN_OVERLAP),
    max_pathway_size=int(PATHWAY_MAX_PATHWAY_SIZE),
    gsea_permutations=int(PATHWAY_GSEA_PERMUTATIONS),
    seed=int(PATHWAY_SEED),
    organism=str(PATHWAY_ORGANISM),
    n_cpus=max(1, int(N_CPUS)),
    progress_callback=pathway_events.append,
)
pathway_enrichment = pseudobulk_de_for_pathway[GROUPBY][source][reference_key].get(
    "pathway_enrichment"
)

print(json.dumps(pathway_settings, indent=2))
print(f"recorded pathway progress events: {len(pathway_events)}")
pathway_enrichment

In [ ]:
# Display compact ORA and GSEA result tables while keeping full rows for plotting.
def pathway_rows_df(method, direction):
    rows = (
        []
        if globals().get("pathway_enrichment") is None
        else pathway_enrichment.get(method, {}).get(direction, [])
    )
    return pd.DataFrame(rows)


def pathway_table(df):
    if df is None or df.empty:
        return df
    cols = [
        col
        for col in [
            "term",
            "direction",
            "padj",
            "pval",
            "odds_ratio",
            "nes",
            "es",
            "overlap",
            "query_size",
            "pathway_size",
            "rank_count",
            "peak_rank",
            "zero_cross_rank",
            "genes",
            "leading_edge",
        ]
        if col in df.columns
    ]
    return df[cols]


ora_up = pathway_rows_df("ora", "up")
ora_down = pathway_rows_df("ora", "down")
gsea_positive = pathway_rows_df("gsea", "positive")
gsea_negative = pathway_rows_df("gsea", "negative")

ora_up_table = pathway_table(ora_up)
ora_down_table = pathway_table(ora_down)
gsea_positive_table = pathway_table(gsea_positive)
gsea_negative_table = pathway_table(gsea_negative)

print(f"ORA up pathways favoring {source}")
display(ora_up_table.head(int(PATHWAY_TOP_N)))
print(f"ORA down pathways favoring {reference_label}")
display(ora_down_table.head(int(PATHWAY_TOP_N)))
print(f"GSEA positive enrichment pathways favoring {source}")
display(gsea_positive_table.head(int(PATHWAY_TOP_N)))
print(f"GSEA negative enrichment pathways favoring {reference_label}")
display(gsea_negative_table.head(int(PATHWAY_TOP_N)))

for name, df in {
    "gsea_positive": gsea_positive,
    "gsea_negative": gsea_negative,
}.items():
    missing = {"running_profile", "hit_indices", "rank_metric_profile"}.difference(
        df.columns if df is not None else []
    )
    if df is not None and not df.empty and missing:
        print(
            f"Warning: {name} is missing {sorted(missing)}. "
            "Rerun the pathway computation cell above, or restart the kernel if an old KaroSpace module was cached."
        )

In [ ]:
# Plot ORA dot plots and classic GSEA enrichment profiles, matching the viewer summaries.
from matplotlib.colors import LinearSegmentedColormap, Normalize


def _short_term(value, max_len=58):
    text = str(value or "")
    return text if len(text) <= max_len else text[: max_len - 1] + "…"


def _empty_axis(ax, title, message="No pathways"):
    ax.axis("off")
    ax.set_title(title)
    ax.text(0.5, 0.5, message, ha="center", va="center", transform=ax.transAxes)


def _score_norm(values):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return Normalize(vmin=0.0, vmax=1.0)
    vmin = float(np.nanmin(arr))
    vmax = float(np.nanmax(arr))
    if np.isclose(vmin, vmax):
        return Normalize(vmin=max(0.0, vmin * 0.5), vmax=vmax * 1.5 + 1e-12)
    return Normalize(vmin=vmin, vmax=vmax)


def plot_ora_dotplot(ax, df, title):
    if df is None or df.empty:
        _empty_axis(ax, title)
        return
    plot_df = df.copy()
    plot_df["overlap_n"] = pd.to_numeric(plot_df["overlap"], errors="coerce").fillna(
        0.0
    )
    plot_df["query_size_n"] = pd.to_numeric(
        plot_df["query_size"], errors="coerce"
    ).replace(0, np.nan)
    query_size = plot_df["query_size_n"].to_numpy(dtype=float)
    plot_df["gene_ratio"] = np.divide(
        plot_df["overlap_n"].to_numpy(dtype=float),
        query_size,
        out=np.zeros(len(plot_df), dtype=float),
        where=np.isfinite(query_size),
    )
    plot_df["padj_n"] = (
        pd.to_numeric(plot_df["padj"], errors="coerce")
        .fillna(1.0)
        .clip(np.nextafter(0.0, 1.0), 1.0)
    )
    plot_df["neg_log10_padj"] = -np.log10(plot_df["padj_n"])
    plot_df = (
        plot_df.sort_values(
            ["gene_ratio", "padj_n", "term"], ascending=[False, True, True]
        )
        .head(16)
        .sort_values("gene_ratio", ascending=True)
    )

    overlap = plot_df["overlap_n"].to_numpy(dtype=float)
    gene_ratio = plot_df["gene_ratio"].to_numpy(dtype=float)
    score = plot_df["neg_log10_padj"].to_numpy(dtype=float)
    size_range = max(float(np.nanmax(overlap) - np.nanmin(overlap)), 1.0)
    sizes = (
        55.0
        + np.sqrt(np.maximum(overlap - np.nanmin(overlap), 0.0) / size_range) * 260.0
    )
    edge_colors = np.where(
        plot_df["padj_n"].to_numpy(dtype=float) < 0.05, "#d94f4f", "#111111"
    )
    y = np.arange(len(plot_df))
    blue_cmap = LinearSegmentedColormap.from_list(
        "ora_blue", ["#d7e8ff", "#5aa7e8", "#0f4f96"]
    )
    scatter = ax.scatter(
        gene_ratio,
        y,
        s=sizes,
        c=score,
        cmap=blue_cmap,
        norm=_score_norm(score),
        edgecolors=edge_colors,
        linewidths=1.8,
        alpha=0.95,
    )
    ax.set_yticks(y)
    ax.set_yticklabels([_short_term(term, 54) for term in plot_df["term"]], fontsize=7)
    ax.set_xlabel("GeneRatio")
    ax.set_title(title)
    ax.grid(axis="x", color="0.9", linewidth=0.8)
    ax.margins(x=0.08)
    cbar = plt.colorbar(scatter, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("-log10 adjusted p")

    unique_sizes = np.unique(overlap.astype(int))
    handles = []
    labels = []
    if unique_sizes.size:
        legend_values = np.unique(
            np.round(
                np.linspace(
                    unique_sizes.min(), unique_sizes.max(), min(3, unique_sizes.size)
                )
            ).astype(int)
        )
        for value in legend_values:
            radius_size = (
                55.0
                + np.sqrt(max(value - np.nanmin(overlap), 0.0) / size_range) * 260.0
            )
            handles.append(
                ax.scatter([], [], s=radius_size, facecolors="none", edgecolors="0.35")
            )
            labels.append(str(int(value)))
        size_legend = ax.legend(
            handles,
            labels,
            title="number of genes",
            loc="lower right",
            frameon=False,
            fontsize=8,
            title_fontsize=8,
            bbox_to_anchor=(1.0, 0.02),
        )
        ax.add_artist(size_legend)

    border_handles = [
        ax.scatter(
            [], [], s=80, facecolors="none", edgecolors="#d94f4f", linewidths=1.8
        ),
        ax.scatter(
            [], [], s=80, facecolors="none", edgecolors="#111111", linewidths=1.8
        ),
    ]
    ax.legend(
        border_handles,
        ["adj. p < 0.05", "adj. p >= 0.05"],
        title="border",
        loc="lower right",
        bbox_to_anchor=(1.0, 0.31),
        frameon=False,
        fontsize=8,
        title_fontsize=8,
    )


def _to_numeric_profile(values):
    if not isinstance(values, (list, tuple)) or len(values) == 0:
        return np.empty((0, 2), dtype=float)
    arr = np.asarray(values, dtype=float)
    if arr.ndim != 2 or arr.shape[1] < 2:
        return np.empty((0, 2), dtype=float)
    return arr[:, :2]


def _scale_to_band(values, low, high, include_zero=True):
    arr = np.asarray(values, dtype=float)
    finite = arr[np.isfinite(arr)]
    if include_zero:
        finite = np.concatenate([finite, np.array([0.0])])
    if finite.size == 0:
        return np.full_like(arr, (low + high) / 2.0, dtype=float)
    vmin = float(np.nanmin(finite))
    vmax = float(np.nanmax(finite))
    if np.isclose(vmin, vmax):
        return np.full_like(arr, (low + high) / 2.0, dtype=float)
    return low + (arr - vmin) / (vmax - vmin) * (high - low)


def _fmt_number(value):
    try:
        number = float(value)
    except (TypeError, ValueError):
        return "n/a"
    return f"{number:.3g}" if np.isfinite(number) else "n/a"


def plot_gsea_enrichment(
    ax, df, title, primary_color="#d94f4f", secondary_color="#4f82d9"
):
    if df is None or df.empty:
        _empty_axis(ax, title)
        return
    row = df.iloc[0].to_dict()
    profile = _to_numeric_profile(row.get("running_profile"))
    metric = _to_numeric_profile(row.get("rank_metric_profile"))
    hits = np.asarray(row.get("hit_indices") or [], dtype=float)
    rank_count = int(row.get("rank_count") or 0)
    if profile.size == 0 or metric.size == 0 or hits.size == 0 or rank_count < 2:
        _empty_axis(ax, title, message="No GSEA profile data")
        return

    direction = str(row.get("direction", "positive"))
    direction_positive = direction != "negative"
    display_sign = 1.0 if direction_positive else -1.0
    profile_values = display_sign * profile[:, 1]
    metric_values = display_sign * metric[:, 1]
    display_profile_x = (
        profile[:, 0] if direction_positive else (rank_count - 1) - profile[:, 0]
    )
    display_metric_x = (
        metric[:, 0] if direction_positive else (rank_count - 1) - metric[:, 0]
    )
    display_hits = hits if direction_positive else (rank_count - 1) - hits
    profile_color = primary_color if direction_positive else secondary_color
    opposite_color = secondary_color if direction_positive else primary_color
    label_a = source if direction_positive else reference
    label_b = reference if direction_positive else source
    term = _short_term(row.get("term"), max_len=60)
    ax.set_title(f"{title}: {term}", fontsize=10)
    ax.set_xlim(0, rank_count - 1)
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_xlabel("Rank in ordered dataset")
    for spine in ["left", "right", "top"]:
        ax.spines[spine].set_visible(False)

    es_low, es_high = 0.63, 0.98
    hit_low, hit_high = 0.45, 0.58
    metric_low, metric_high = 0.08, 0.38
    for low, high in [
        (es_low, es_high),
        (hit_low, hit_high),
        (metric_low, metric_high),
    ]:
        ax.axhspan(low, high, color="none", ec="0.82", lw=0.8)
        for frac in [0.25, 0.5, 0.75]:
            ax.axhline(
                low + (high - low) * frac, color="0.9", lw=0.7, ls="--", zorder=0
            )

    es_y = _scale_to_band(profile_values, es_low, es_high)
    zero_es = _scale_to_band(np.array([0.0]), es_low, es_high)[0]
    ax.axhline(zero_es, color="0.65", lw=0.8)
    profile_order = np.argsort(display_profile_x)
    ax.plot(
        display_profile_x[profile_order],
        es_y[profile_order],
        color=profile_color,
        lw=2.2,
        solid_joinstyle="round",
    )
    peak_rank = row.get("peak_rank")
    if pd.notna(peak_rank):
        display_peak = (
            float(peak_rank)
            if direction_positive
            else (rank_count - 1) - float(peak_rank)
        )
        ax.axvline(
            display_peak,
            ymin=es_low,
            ymax=es_high,
            color=profile_color,
            lw=0.9,
            ls="--",
            alpha=0.35,
        )

    for hit in display_hits:
        ax.vlines(hit, hit_low + 0.01, hit_high - 0.01, color="black", lw=0.8)
    cmap = LinearSegmentedColormap.from_list(
        "gsea_direction", [profile_color, "#f5f5f5", opposite_color]
    )
    ax.imshow(
        np.linspace(0, 1, 256).reshape(1, -1),
        extent=[0, rank_count - 1, hit_low - 0.035, hit_low - 0.005],
        aspect="auto",
        cmap=cmap,
        interpolation="nearest",
    )

    metric_y = _scale_to_band(metric_values, metric_low, metric_high)
    zero_metric = _scale_to_band(np.array([0.0]), metric_low, metric_high)[0]
    ax.axhline(zero_metric, color="0.65", lw=0.8)
    metric_order = np.argsort(display_metric_x)
    ax.vlines(
        display_metric_x[metric_order],
        zero_metric,
        metric_y[metric_order],
        color="0.72",
        lw=1.2,
        alpha=0.9,
    )
    zero_cross = row.get("zero_cross_rank")
    if pd.notna(zero_cross):
        display_zero = (
            float(zero_cross)
            if direction_positive
            else (rank_count - 1) - float(zero_cross)
        )
        ax.axvline(
            display_zero,
            ymin=metric_low,
            ymax=metric_high,
            color="0.5",
            lw=0.9,
            ls="--",
        )
        ax.text(
            display_zero,
            metric_low + 0.02,
            "Zero cross",
            ha="center",
            va="bottom",
            fontsize=8,
            color="0.35",
        )

    ax.text(
        -0.03,
        (es_low + es_high) / 2,
        "ES",
        ha="right",
        va="center",
        rotation=90,
        transform=ax.transAxes,
        fontsize=9,
    )
    ax.text(
        -0.03,
        (metric_low + metric_high) / 2,
        "Rank metric",
        ha="right",
        va="center",
        rotation=90,
        transform=ax.transAxes,
        fontsize=9,
    )
    ax.text(
        0.01,
        hit_low - 0.055,
        f"{label_a} enriched",
        color=profile_color,
        ha="left",
        va="top",
        transform=ax.transAxes,
        fontsize=8,
    )
    ax.text(
        0.99,
        hit_low - 0.055,
        f"{label_b} enriched",
        color=opposite_color,
        ha="right",
        va="top",
        transform=ax.transAxes,
        fontsize=8,
    )
    ax.text(
        0.01,
        0.01,
        f"NES={_fmt_number(abs(row.get('nes')))}; ES={_fmt_number(abs(row.get('es')))}; adj. p={_fmt_number(row.get('padj'))}",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=8,
        color="0.35",
    )


fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), constrained_layout=True)
plot_ora_dotplot(axes[0], ora_up, f"ORA up: {source}")
plot_ora_dotplot(axes[1], ora_down, f"ORA down: {reference_label}")
plt.show()


def _gsea_dropdown_rows():
    frames = []
    if gsea_positive is not None and not gsea_positive.empty:
        frames.append(gsea_positive.assign(gsea_direction_label=f"{source} enriched"))
    if gsea_negative is not None and not gsea_negative.empty:
        frames.append(
            gsea_negative.assign(gsea_direction_label=f"{reference_label} enriched")
        )
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


gsea_dropdown_df = _gsea_dropdown_rows()


def _plot_selected_gsea_pathway(row_index):
    row_index = int(row_index)
    selected = gsea_dropdown_df.iloc[[row_index]]
    fig, ax = plt.subplots(1, 1, figsize=(12, 6), constrained_layout=True)
    direction = selected.iloc[0].get("gsea_direction_label", "GSEA")
    plot_gsea_enrichment(
        ax, selected, str(direction), primary_color="#d94f4f", secondary_color="#4f82d9"
    )
    plt.show()


if gsea_dropdown_df.empty:
    print("No GSEA pathways available for the dropdown.")
else:
    options = []
    for idx, row in gsea_dropdown_df.iterrows():
        nes = _fmt_number(abs(row.get("nes")))
        padj = _fmt_number(row.get("padj"))
        label = f"{row.get('gsea_direction_label')}: {_short_term(row.get('term'), 72)} | NES={nes}, adj. p={padj}"
        options.append((label, int(idx)))
    try:
        import ipywidgets as widgets
        from IPython.display import display

        selector = widgets.Dropdown(
            options=options, description="Pathway:", layout=widgets.Layout(width="100%")
        )
        output = widgets.interactive_output(
            _plot_selected_gsea_pathway, {"row_index": selector}
        )
        display(widgets.VBox([selector, output]))
    except Exception as exc:
        print(
            "ipywidgets is unavailable; showing the first retained GSEA pathway instead."
        )
        print(exc)
        _plot_selected_gsea_pathway(options[0][1])

In [ ]:
# Optional: save pathway tables next to the audited DE table.
# pathway_prefix = Path(H5AD_PATH).with_suffix(f".{GROUPBY}.{source}_vs_{reference_key}.pathway")
# for name, df in {
#     "ora_up": ora_up_table,
#     "ora_down": ora_down_table,
#     "gsea_positive": gsea_positive_table,
#     "gsea_negative": gsea_negative_table,
# }.items():
#     if df is not None and not df.empty:
#         out_path = pathway_prefix.with_suffix(f".{name}.csv")
#         df.to_csv(out_path, index=False)
#         print(out_path)